# 02 Consumer Experience Intelligence

**Research question:** How are delivery performance and freight costs associated with customer review scores?
**Sources:** `analytics.fact_reviews` (review grain, 1 row/review_id), `analytics.fact_delivery` (order grain). **Filter:** `valid_review_score IS NOT NULL`, delivered orders for delivery metrics.

In [1]:
from src.db import get_connection

conn = get_connection(read_only=True)


In [2]:
# Grain: 1 row per review joined to its order delivery (order grain)
by_bucket = conn.execute("""
SELECT fd.delivery_time_bucket,
       COUNT(*) AS reviews,
       AVG(fr.valid_review_score) AS avg_review
FROM analytics.fact_reviews fr
JOIN analytics.fact_delivery fd ON fr.order_id = fd.order_id
WHERE fr.valid_review_score IS NOT NULL
  AND fd.delivery_time_bucket IS NOT NULL
GROUP BY fd.delivery_time_bucket
ORDER BY fd.delivery_time_bucket
""").fetchdf()
by_bucket

,delivery_time_bucket,reviews,avg_review
0,0-3 days,6914,4.462829
1,15-30 days,23256,3.940961
2,30+ days,4169,2.202207
3,4-7 days,23584,4.398660
4,8-14 days,37684,4.300870


In [3]:
# On-time vs late (grain: review/order join). Late = delivery_gap_days < 0
timing = conn.execute("""
SELECT CASE WHEN fd.delivery_gap_days >= 0 THEN 'on_time_or_early' ELSE 'late' END AS timing,
       COUNT(*) AS reviews,
       AVG(fr.valid_review_score) AS avg_review
FROM analytics.fact_reviews fr
JOIN analytics.fact_delivery fd ON fr.order_id = fd.order_id
WHERE fr.valid_review_score IS NOT NULL AND fd.delivery_gap_days IS NOT NULL
GROUP BY 1
""").fetchdf()
timing

,timing,reviews,avg_review
0,on_time_or_early,89245,4.292207
1,late,6362,2.270041


In [4]:
# Spearman correlations (review grain). Associations, not causation.
corr = conn.execute("""
SELECT corr(fr.valid_review_score, fd.delivery_time_days) AS corr_delivery_days,
       corr(fr.valid_review_score, fd.delivery_gap_days) AS corr_gap_days,
       corr(fr.valid_review_score, fd.order_freight_value) AS corr_freight_value,
       corr(fr.valid_review_score, fd.freight_share) AS corr_freight_share
FROM analytics.fact_reviews fr
JOIN analytics.fact_delivery fd ON fr.order_id = fd.order_id
WHERE fr.valid_review_score IS NOT NULL AND fd.is_delivered
""").fetchdf()
corr

,corr_delivery_days,corr_gap_days,corr_freight_value,corr_freight_share
0,-0.334278,0.267314,-0.089874,-0.026707
